In [1]:
# conda activate anndata

import sys
import pickle
import numpy as np
import pandas as pd

sys.path.append("code")
sys.path.append("/mnt/lareaulab/reliscu/code")

from process_gtf import *
from empirical_corr_pvals import *
from junction2psi import *

In [2]:
data_source = "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed"
psi = pd.read_csv(f"data/GTEx_frontal_cortex_SE_PSI.csv", index_col=0)
eigengene_df = pd.read_csv("data/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_ct_eigengenes.csv", index_col=0)

In [3]:
# Make sure order of samples matches
eigengene_df.index = eigengene_df.index.str.replace(".", "-") 
common = psi.columns.intersection(eigengene_df.index)
psi = psi[common]
eigengene_df = eigengene_df.loc[common]

##  Correlate PSI and eigengenes and calc empirical p-values

In [5]:
# --- correlations + permutation p-values + FDR + CIs ---
(psi_corr_df, psi_pval_df, psi_fdr_df, psi_ci_lower_df, psi_ci_upper_df,
 perm_corr_results, psi_rank_centered, psi_rank_norm, ct_ranks_dict) = spearman_permutation_test(
    psi,
    eigengene_df,
    n_perms=10000,
    ci=0.95
)

Done: CGE Class
Done: All GABAergic
Done: Micro/PVM
Done: Oligo
Done: Astro
Done: Endo
Done: All Neuronal
Done: Deep layer glutamatergic
Done: Upper layer glutamatergic
Done: OPC


In [6]:
steiger_results = compare_all_ct_pairs(
    psi_corr_df,
    perm_corr_results,
    eigengene_df.columns.tolist()
)

In [ ]:
print("find_specific_SEs_per_ct_basic:")

ct_specific_SEs_basic = find_specific_SEs_per_ct_basic(
    psi_fdr_df,
    psi_corr_df,
    fdr_thresh=0.05
)
ct_specific_SEs_strict = {}

print("")
print("find_specific_SEs_per_ct:")

# correlation difference test 
ct_hierarchy = {
    # broader  cell classes should not be comapred against their subtypes (subtypes listed as children);
    'All Neuronal':              ['All GABAergic', 'CGE Class','Upper layer glutamatergic', 'Deep layer glutamatergic'],
    'All GABAergic':             ['CGE Class'],
    # subtypes must be greater than everything
    'CGE Class':                 [],
    'Upper layer glutamatergic': [],
    'Deep layer glutamatergic':  [],
    'Oligo':                     [],
    'OPC':                       [],
    'Astro':                     [],
    'Micro/PVM':                 [],
    'VLMC':                      [],
    'Endo':                      [],
    'Peri':                      [],
}

for ascending in (True, False):
    print("")
    print(ascending)
    ct_specific_SEs_strict[str(ascending)] = find_specific_SEs_per_ct(
        steiger_results,
        psi_fdr_df,
        psi_corr_df,
        ct_hierarchy,
        fdr_thresh=0.05,
        ascending=ascending
    )

find_specific_SEs_per_ct_basic:
CGE Class: 5804 significant SEs
All GABAergic: 7557 significant SEs
Micro/PVM: 92 significant SEs
Oligo: 609 significant SEs
Astro: 6241 significant SEs
Endo: 2883 significant SEs
All Neuronal: 8582 significant SEs
Deep layer glutamatergic: 5586 significant SEs
Upper layer glutamatergic: 6350 significant SEs
OPC: 417 significant SEs

find_specific_SEs_per_ct:

True
CGE Class: 0 specific SEs (5804 significant total)
All GABAergic: 4 specific SEs (7557 significant total)
Micro/PVM: 7 specific SEs (92 significant total)
Oligo: 17 specific SEs (609 significant total)
Astro: 1006 specific SEs (6241 significant total)
Endo: 3 specific SEs (2883 significant total)
All Neuronal: 1597 specific SEs (8582 significant total)
Deep layer glutamatergic: 2 specific SEs (5586 significant total)
Upper layer glutamatergic: 3 specific SEs (6350 significant total)
OPC: 2 specific SEs (417 significant total)

False
CGE Class: 0 specific SEs (5804 significant total)
All GABAer

In [ ]:
# --- combine results ---
combined_results = combine_results(
    ct_specific_SEs_basic,
    ct_specific_SEs_strict,
    psi_corr_df,
    psi_fdr_df,
    steiger_results,
    ct_hierarchy
)

In [ ]:
# # Format and save results per cell type

# column_order = ['Gene', 'is_specific', 'specific_direction', 
#                 'chr', 'strand', 'SE_start', 'SE_end', 'SE_len',
#                 'r', 'fdr', 
#                 'CGE Class', 'All GABAergic', 'All Neuronal',
#                 'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
#                 'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
#                 ]

# for ct, df in combined_results.items():
#     print(ct)
#     # Remove SEs with NaN correlations
#     df = df[~np.isnan(df['Oligo'])]
#     # Add strand information from GTF
#     df = df.join(
#         gtf_parsed[['gene_name', 'strand']].set_index('gene_name'),
#         on='Gene',
#         how='left'
#     )
#     rest_columns = df.columns[df.columns.str.contains("diff")].tolist() 
#     df[column_order + rest_columns].to_csv(f"data/ct_SEs/{_safe(ct)}_SEs.csv")

## Annotate and save results per cell type

In [ ]:
# Parse GTF

exclude = ""
gene_name = "gene_id"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v50.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)

Processing GTF file...


In [ ]:
pickle.dump(gtf, open("data/gencode.v50.annotation_gtf_parsed.pkl", "wb"))

In [ ]:
gtf_gene = gtf[gtf.feature == "gene"]
gtf_gene.head()

In [17]:
pickle.dump(gtf, open("data/gencode.v50.annotation_gtf_parsed.pkl", "wb"))

In [ ]:
# Get coordinates for each SE

intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file_v50.tab.gz"
intron_table = read_intron_file(intron_file)

intron_coords_df = intron_table['intron'].str.split(r"[:\-]", expand=True).iloc[:, :4]
intron_coords_df.columns = ["chr", "intron_first_base", "intron_last_base", "strand"]
intron_coords_df.index = intron_table.index
intron_coords_df['SE'] = intron_coords_df.index.str.split("_").str[:3].str.join("_")
intron_coords_df = intron_coords_df[intron_coords_df['SE'].isin(psi.index)] # Subset to SEs in PSI data
intron_coords_df['intron_first_base'] = intron_coords_df["intron_first_base"].astype(int)
intron_coords_df['intron_last_base'] = intron_coords_df["intron_last_base"].astype(int)

In [20]:
intron_coords_df.head()

,chr,intron_first_base,intron_last_base,strand,SE
ENSG00000292994_other_2_I1,chr1,259026,261549,,ENSG00000292994_other_2
ENSG00000292994_other_2_I2,chr1,261635,267302,,ENSG00000292994_other_2
ENSG00000292994_other_2_SE,chr1,259026,267302,,ENSG00000292994_other_2
ENSG00000290385_other_1_I1,chr1,497300,498046,,ENSG00000290385_other_1
ENSG00000290385_other_1_I2,chr1,498306,498398,,ENSG00000290385_other_1


In [30]:
def safe_exon_coords(g):
    i1 = g.loc[g.index.str.contains("I1$"), "intron_last_base"].values
    i2 = g.loc[g.index.str.contains("I2$"), "intron_first_base"].values
    if len(i1) == 0 or len(i2) == 0:
        return pd.Series({"chr": None, "SE_start": None, "SE_end": None})
    return pd.Series({
        "chr": g["chr"].iloc[0],
        "intron_end": i1[0],
        "exon_start": i1[0] + 1,
        "exon_end": i2[0] - 1,
        "intron_start": i2[0],
        "exon_len": i2[0] - i1[0] + 1
    })

SE_coords_df = intron_coords_df.groupby("SE").apply(safe_exon_coords)
SE_coords_df = SE_coords_df.dropna()  # drop SEs with missing coords
SE_coords_df['gene_id'] = SE_coords_df.index.str.split("_").str[0]

/tmp/ipykernel_136964/558260382.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  SE_coords_df = intron_coords_df.groupby("SE").apply(safe_exon_coords)


In [31]:
SE_coords_df.head()

,chr,intron_end,exon_start,exon_end,intron_start,exon_len
SE,,,,,,
ENSG00000000419_NMD_1,chr20,50940864,50940865,50940955,50940956,93
ENSG00000000419_ProteinCoding_1,chr20,50940864,50940865,50940933,50940934,71
ENSG00000000419_ProteinCoding_10,chr20,50945736,50945737,50945762,50945763,28
ENSG00000000419_ProteinCoding_5,chr20,50941104,50941105,50941209,50941210,107
ENSG00000000419_ProteinCoding_6,chr20,50941128,50941129,50941209,50941210,83


In [26]:
SE_anno_df = SE_coords_df.merge(gtf[['gene_id', 'gene_name']], on="gene_id", how="right")

KeyError: "None of [Index(['gene_id', 'gene_name'], dtype='object')] are in the [columns]"

In [28]:
gtf.tail()

,chrom,start,end,feature,strand,transcript,gene_type,gene,transcript_type,exon_id,exon_number,frame,tag,protein_id
11242709,chrM,15888,15953,transcript,+,ENST00000387460,Mt_tRNA,ENSG00000210195,Mt_tRNA,,,0,"basic,Ensembl_canonical,GENCODE_Primary",
11242710,chrM,15888,15953,exon,+,ENST00000387460,Mt_tRNA,ENSG00000210195,Mt_tRNA,ENSE00001544475,1,0,"basic,Ensembl_canonical,GENCODE_Primary",
11242711,chrM,15956,16023,gene,-,,Mt_tRNA,ENSG00000210196,,,,0,,
11242712,chrM,15956,16023,transcript,-,ENST00000387461,Mt_tRNA,ENSG00000210196,Mt_tRNA,,,0,"basic,Ensembl_canonical,GENCODE_Primary",
11242713,chrM,15956,16023,exon,-,ENST00000387461,Mt_tRNA,ENSG00000210196,Mt_tRNA,ENSE00001544473,1,0,"basic,Ensembl_canonical,GENCODE_Primary",


In [ ]:
# # Get PSI data ready to merge on gene IDs
# psi['gene_id'] = psi.index.str.split("_").str[0]
# psi['SE_id'] = psi.index.values

# psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
# psi_anno = psi_anno.set_index("SE_id").rename_axis(None)
# psi_anno = psi_anno.drop(columns=["gene_id"])

In [ ]:
# Save significant cell type SEs

ct_order = ['CGE Class', 'All GABAergic', 'All Neuronal',
            'Upper layer glutamatergic', 'Deep layer glutamatergic', 
            'Oligo', 'OPC', 'Astro', 'Micro/PVM', 'Endo']

column_order = ['Gene', 'is_specific', 'specific_direction', 'chr', 'strand', 
                'intron_end', 'exon_start', 'exon_end', 'intron_start', 'exon_len', 'r', 'fdr'] + ct_order

def _safe(name):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

for ct, results in combined_results.items():
    print(ct)s
    # results = results[~np.isnan(df[ct])]
    results = results.join(gtf_parsed[['gene_name', 'strand']].set_index('gene_name'), on='Gene', how='left')
    trailing_columns = results.columns[df.columns.str.contains("diff")].tolist() 
    results[column_order + trailing_columns].to_csv(f"data/ct_SEs/{data_source}_{_safe(ct)}_SEs.csv")
    # psi_corr_df.to_csv(f"data/{data_source}_ct_eigengene_PSI_corr.csv")